In [ ]:
import os
import glob
import numpy as np
import flammkuchen as fl
import tifffile
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from tqdm import tqdm  

from pathlib import Path

In [ ]:
def calculate_dff(data, baseline_mask, sigma_time=None, sigma_space=None, correct_offset=True):
    """
    Calculate ΔF/F for calcium imaging data with optional offset correction.
    
    Parameters:
    -----------
    data : numpy.ndarray
        3D array of fluorescence data (t, y, x)
    baseline_mask : numpy.ndarray
        1D boolean array indicating which timepoints to use for baseline (True = use for baseline)
    sigma_time : float or None
        Standard deviation for Gaussian smoothing along time axis (None = no smoothing)
    sigma_space : float or None
        Standard deviation for Gaussian smoothing along spatial axes (None = no smoothing)
    correct_offset : bool
        Whether to estimate and correct for any DC offset in the data
    
    Returns:
    --------
    tuple
        (dff, offset_correction)
    """
    # Check if we have baseline timepoints
    if not np.any(baseline_mask):
        print("  Warning: No baseline timepoints found in the mask. Using all timepoints.")
        baseline_mask = np.ones_like(baseline_mask, dtype=bool)
    
    # Optional: Smooth the data
    smoothed_data = data.copy()
    if sigma_time is not None or sigma_space is not None:
        sigma = (sigma_time or 0, sigma_space or 0, sigma_space or 0)
        smoothed_data = gaussian_filter(data, sigma=sigma)
    
    # Offset correction
    offset_correction = None
    if correct_offset:
        # Estimate offset using 5th percentile
        print("  Estimating offset correction...")
        offset_correction = np.percentile(smoothed_data, 5, axis=0, keepdims=True)
        
        # Apply offset correction
        smoothed_data = smoothed_data - offset_correction
        
        # Ensure no negative values
        smoothed_data = np.maximum(smoothed_data, 0)
        
        print(f"  Applied offset correction: min={np.min(offset_correction):.2f}, max={np.max(offset_correction):.2f}")
    
    # Calculate baseline (F0) as the mean of non-motion periods
    F0 = np.mean(smoothed_data[baseline_mask, :, :], axis=0, keepdims=True)
    
    # Avoid division by zero
    min_value = np.min(F0[F0 > 0]) if np.any(F0 > 0) else 1.0
    F0 = np.where(F0 > 0, F0, min_value)
    
    # Calculate ΔF/F
    dff = (smoothed_data - F0) / F0
    
    return dff, offset_correction

def create_dff_qc_figure(raw_data, dff_data, baseline_mask, offset_correction=None, output_path='dff_qc.png'):
    """Create a quality control figure."""
    # Calculate mean across spatial dimensions
    mean_raw = np.mean(raw_data, axis=(1, 2))
    mean_dff = np.mean(dff_data, axis=(1, 2))
    
    # Create figure
    plt.figure(figsize=(10, 10))
    
    # Plot baseline mask
    plt.subplot(4, 1, 1)
    plt.plot(baseline_mask.astype(int), 'k-')
    plt.title("Baseline Mask (Non-motion Periods)")
    plt.ylabel("Baseline")
    
    # Plot raw fluorescence
    plt.subplot(4, 1, 2)
    plt.plot(mean_raw, 'g-')
    plt.title("Raw Fluorescence (Spatial Mean)")
    plt.ylabel("Fluorescence")
    
    # Plot offset-corrected data if available
    if offset_correction is not None:
        # Calculate mean of offset-corrected data
        offset_corrected = raw_data - offset_correction
        mean_corrected = np.mean(offset_corrected, axis=(1, 2))
        
        plt.subplot(4, 1, 3)
        plt.plot(mean_corrected, 'c-')
        plt.title("Offset-Corrected Fluorescence (Spatial Mean)")
        plt.ylabel("Fluorescence")
        
        # Plot ΔF/F
        plt.subplot(4, 1, 4)
    else:
        # Plot ΔF/F without offset subplot
        plt.subplot(4, 1, 3)
    
    plt.plot(mean_dff, 'b-')
    plt.title("ΔF/F (Spatial Mean)")
    plt.ylabel("ΔF/F")
    plt.xlabel("Time (frames)")
    
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

def process_session_dff(session_folder, sigma_time=1.0, sigma_space=0.0, correct_offset=True):
    """Process a single session folder to calculate ΔF/F."""
    session_name = os.path.basename(session_folder)
    print(f"\nProcessing session: {session_name}")
    
    # Paths
    regressor_path = os.path.join(session_folder, "motion_regressors.h5")
    data_file = os.path.join(session_folder, f"{session_name}.h5")
    output_tiff = os.path.join(session_folder, "dff_data.tif")
    output_h5 = os.path.join(session_folder, "dff_metadata.h5")  # Renamed to avoid confusion
    
    # Check files exist
    if not os.path.exists(regressor_path):
        print(f"  No regressor file found at {regressor_path}")
        return
    
    if not os.path.exists(data_file):
        print(f"  No data file found at {data_file}")
        return
    
    # Load regressors
    try:
        regressors = fl.load(regressor_path)
        left_regressor = regressors['left_regressor']
        right_regressor = regressors['right_regressor']
        
        # Create baseline mask (when dots are not moving)
        baseline_mask = ~(left_regressor.astype(bool) | right_regressor.astype(bool))
        
        n_baseline = np.sum(baseline_mask)
        print(f"  Found {n_baseline} baseline timepoints for ΔF/F calculation")
    except Exception as e:
        print(f"  Error loading regressors: {str(e)}")
        return
    
    # Load imaging data
    try:
        print(f"  Loading calcium imaging data from {session_name}.h5")
        img = fl.load(Path(session_folder) / f"{session_name}.h5")['stack_4D']
        
        # Handle 4D data with singleton dimension
        if len(img.shape) == 4 and 1 in img.shape:
            data = np.squeeze(img)
            print(f"  Squeezed 4D data to shape: {data.shape}")
        else:
            data = img
            print(f"  Loaded data with shape: {data.shape}")
        
        # Adjust mask length if needed
        if data.shape[0] != len(baseline_mask):
            print(f"  Adjusting mask length from {len(baseline_mask)} to {data.shape[0]}")
            if data.shape[0] < len(baseline_mask):
                baseline_mask = baseline_mask[:data.shape[0]]
            else:
                baseline_mask = np.pad(baseline_mask, (0, data.shape[0] - len(baseline_mask)), 
                                      mode='constant', constant_values=False)
    except Exception as e:
        print(f"  Error loading data file: {str(e)}")
        return
    
    # Calculate ΔF/F
    print("  Calculating ΔF/F...")
    dff, offset_correction = calculate_dff(
        data, 
        baseline_mask, 
        sigma_time=sigma_time if sigma_time > 0 else None, 
        sigma_space=sigma_space if sigma_space > 0 else None,
        correct_offset=correct_offset
    )
    
    # Save ΔF/F data as TIFF
    print(f"  Saving DFF data to {os.path.basename(output_tiff)}")
    try:
        # Ensure data is float32
        dff_float32 = dff.astype(np.float32)
        
        # Save as TIFF
        tifffile.imwrite(output_tiff, dff_float32)
        
        # Verify the file was created
        if os.path.exists(output_tiff):
            file_size = os.path.getsize(output_tiff) / (1024 * 1024)  # Size in MB
            print(f"  TIFF file saved successfully. Size: {file_size:.2f} MB")
        else:
            print(f"  Warning: TIFF file was not created")
            
    except Exception as e:
        print(f"  Error saving TIFF file: {str(e)}")
        return
    
    # Save metadata separately
    print(f"  Saving metadata to {os.path.basename(output_h5)}")
    try:
        metadata = {
            'baseline_mask': baseline_mask,
            'metadata': {
                'sigma_time': sigma_time,
                'sigma_space': sigma_space,
                'baseline_periods': int(np.sum(baseline_mask))
            }
        }
        
        # Add offset correction if used
        if correct_offset:
            metadata['offset_correction'] = offset_correction
            
        fl.save(output_h5, metadata)
    except Exception as e:
        print(f"  Error saving metadata: {str(e)}")
    
    # Create quality control figure
    create_dff_qc_figure(data, dff, baseline_mask, offset_correction,
                         os.path.join(session_folder, 'dff_qc.png'))
    
    print("  ΔF/F processing complete")

def process_all_sessions(base_dir, session_pattern="00*", sigma_time=1.0, sigma_space=0.0, 
                       correct_offset=True):
    """Process all session folders."""
    # Find all session folders
    session_folders = sorted(glob.glob(os.path.join(base_dir, session_pattern)))
    
    if not session_folders:
        print(f"No session folders found matching pattern '{session_pattern}' in {base_dir}")
        return
    
    print(f"Found {len(session_folders)} session folders")
    
    # Process each session
    for session_folder in session_folders:
        try:
            process_session_dff(session_folder, sigma_time, sigma_space, correct_offset)
        except Exception as e:
            print(f"Error processing session {os.path.basename(session_folder)}: {str(e)}")
            continue

In [ ]:
from glob import glob

In [ ]:
master = Path(r"Z:\Hagar\main\e0020 imaging")

fish_list = list(master.glob("*_v41*"))
fish = fish_list[6]
print(fish)
num_fish = len(fish_list)

In [ ]:
import glob

In [ ]:
for fish in fish_list[6:]:
    base_dir = str(fish / 'suite2p')
    
    process_all_sessions(
            base_dir,
            session_pattern="000*",  # Pattern for session folders
            sigma_time=1.0,          # Temporal smoothing (set to 0 to disable)
            sigma_space=0.0,         # Spatial smoothing (set to 0 to disable)
            correct_offset=True,     # Apply offset correction
        )

In [ ]:
img = fl.load(Path(base_dir) / '0000' / '0000.h5')['stack_4D']